# Titanic: Pipeline и подбор гиперпараметров

Целевая переменная — `survived` (1 — выжил, 0 — не выжил). [Источник данных](https://github.com/mwaskom/seaborn-data/blob/master/titanic.csv).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.validation import check_is_fitted

In [2]:
data_path = Path("data/external/titanic.csv")
if not data_path.exists():
    data_path = Path("../data/external/titanic.csv")
data = pd.read_csv(data_path)

numeric_features = ["age", "sibsp", "parch", "fare"]
categorical_features = ["pclass", "sex", "embarked", "deck"]
features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked", "deck"]
X = data[features].copy()
y = data["survived"]

display(X.head())
display(X.isna().sum().rename("Пропуски").to_frame())

,pclass,sex,age,sibsp,parch,fare,embarked,deck
0,3,male,22.0,1,0,7.2500,S,NaN
1,1,female,38.0,1,0,71.2833,C,C
2,3,female,26.0,0,0,7.9250,S,NaN
3,1,female,35.0,1,0,53.1000,S,C
4,3,male,35.0,0,0,8.0500,S,NaN


,Пропуски
pclass,0
sex,0
age,177
sibsp,0
parch,0
fare,0
embarked,2
deck,688


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)
pd.DataFrame({"Выборка": ["train", "test"], "Строки": [len(X_train), len(X_test)]})

,Выборка,Строки
0,train,712
1,test,179


In [4]:
class MissingCountAdder(TransformerMixin, BaseEstimator):
    def fit(self, X, y=None):
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        return self

    def transform(self, X):
        check_is_fitted(self, "feature_names_in_")
        if list(X.columns) != list(self.feature_names_in_):
            raise ValueError("Набор признаков отличается от обучающего.")
        result = X.copy()
        result["missing_count"] = X.isna().sum(axis=1)
        return result

In [5]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features + ["missing_count"]),
    ("cat", categorical_pipeline, categorical_features),
])
pipeline = Pipeline([
    ("missing_count", MissingCountAdder()),
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=2000, random_state=42)),
])

In [6]:
param_grid = {
    "classifier__C": [0.1, 1.0, 10.0],
    "classifier__class_weight": [None, "balanced"],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(
    pipeline, param_grid, scoring="roc_auc", cv=cv,
    n_jobs=1, refit=True, error_score="raise",
)
search.fit(X_train, y_train)
print("best_params_:", search.best_params_)
print(f"Лучший ROC AUC на CV: {search.best_score_:.4f}")

best_params_: {'classifier__C': 0.1, 'classifier__class_weight': None}
Лучший ROC AUC на CV: 0.8603


In [7]:
prediction = search.predict(X_test)
probability = search.predict_proba(X_test)[:, 1]
metrics = pd.Series({
    "Accuracy": accuracy_score(y_test, prediction),
    "Precision": precision_score(y_test, prediction, zero_division=0),
    "Recall": recall_score(y_test, prediction, zero_division=0),
    "F1": f1_score(y_test, prediction, zero_division=0),
    "ROC AUC": roc_auc_score(y_test, probability),
}, name="Test")
display(metrics.to_frame().round(4))
display(pd.DataFrame(
    confusion_matrix(y_test, prediction, labels=[0, 1]),
    index=pd.Index([0, 1], name="Истинный класс"),
    columns=pd.Index([0, 1], name="Предсказанный класс"),
))

,Test
Accuracy,0.8156
Precision,0.8214
Recall,0.6667
F1,0.7360
ROC AUC,0.8422


Предсказанный класс,0,1
Истинный класс,,
0,100,10
1,23,46


На тестовой выборке accuracy составляет 0,816, а ROC AUC — 0,842; лучший вариант использует `C=0.1` и `class_weight=None`. Подсчёт пропусков, заполнение, масштабирование и кодирование входят в Pipeline, поэтому при подборе параметров преобразования обучаются только на обучающей части каждого фолда.